# Optimizers in deep learning: why plain gradient descent is not enough
The 9 parameters of a small network, the number of updates of each gradient descent variant,
and plain gradient descent on a narrow valley with different learning rates.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import keras

## A network with 2 inputs, one hidden layer of 2 nodes and 1 output: 9 parameters

In [ ]:
model = keras.Sequential([keras.Input(shape=(2,)),
                          keras.layers.Dense(2, activation="relu"),
                          keras.layers.Dense(1, activation="sigmoid")])
print("parameters:", model.count_params())    # 2*2 + 2 + 2*1 + 1

## Number of weight updates: 500 rows, 10 epochs

In [ ]:
rows, epochs = 500, 10
for name, batch in (("batch", rows), ("stochastic", 1), ("mini-batch of 100", 100)):
    print(f"{name:18s}: {epochs * rows // batch} updates")

## Plain gradient descent on a narrow valley L = (w1^2 + 100 w2^2) / 2
Start at (-10, 0.4); count the steps until the loss is below 0.01.

In [ ]:
K = 100
loss = lambda w: 0.5 * (w[0] ** 2 + K * w[1] ** 2)
grad = lambda w: np.array([w[0], K * w[1]])

def descend(eta, steps=5000):
    w = np.array([-10.0, 0.4])
    for k in range(1, steps + 1):
        w = w - eta * grad(w)
        if not np.isfinite(loss(w)) or loss(w) > 1e6:
            return None, w                  # diverged
        if loss(w) < 0.01:
            return k, w
    return None, w

for eta in (0.002, 0.005, 0.01, 0.019, 0.02, 0.021):
    k, w = descend(eta)
    print(f"eta {eta:<6}: " + (f"{k} steps" if k else "never (diverges or stalls)"))

Where each run is after 50 steps (the three panels of the figure).

In [ ]:
for eta in (0.002, 0.019, 0.021):
    w = np.array([-10.0, 0.4])
    for _ in range(50):
        w = w - eta * grad(w)
    print(f"eta {eta}: w1 = {w[0]:.2f}, w2 = {w[1]:.3g}, loss = {loss(w):.3g}")

Each step multiplies w1 by (1 - eta) and w2 by (1 - 100 eta): w2 shrinks only if |1 - 100 eta| < 1, that is eta < 0.02.

In [ ]:
for eta in (0.002, 0.019, 0.021):
    print(f"eta {eta}: factor on w1 = {1 - eta:.3f}, factor on w2 = {1 - K * eta:.3f}")